# Module 5 : 7 façons de classer un texte, comparées *(optionnel)*

On reprend la tâche du **J3** (deviner si une critique Allociné est *positive* ou *négative*) et on met en concurrence **7 approches**, du plus simple au plus sophistiqué. Toutes sont évaluées de la même façon : **accuracy** et **F1** (vus au J2) sur le même jeu de test.

| # | Approche | Idée |
|---|---|---|
| 1 | Comptes de mots + régression logistique | sac de mots brut |
| 2 | TF-IDF (vocabulaire nettoyé) + régression logistique | mots pondérés, vocabulaire réduit |
| 3 | TF-IDF (vocabulaire nettoyé) + random forest | idem, autre classifieur |
| 4 | DistilCamemBERT **gelé** + random forest | embeddings d'un transformer, sans adaptation |
| 5 | DistilCamemBERT **fine-tuning partiel** + random forest | on dégèle les couches hautes |
| 6 | DistilCamemBERT **fine-tuning complet** + random forest | on adapte tout l'encodeur |
| 7 | **LLM** *chain-of-thought* (gpt-4o-mini) | on demande au modèle de raisonner puis trancher |

> Notebook **optionnel** et avancé. Active le **GPU** (Colab : `Exécution` > `Modifier le type d'exécution` > **GPU T4**). Le modèle 7 nécessite une **clé OpenAI** (communiquée en classe).

In [ ]:
# Sur Colab : installation des paquets (datasets, spacy et openai ne sont pas toujours préinstallés)
!pip install -q transformers datasets spacy openai
!python -m spacy download fr_core_news_sm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import accuracy_score, f1_score

materiel = "GPU disponible" if torch.cuda.is_available() else "CPU seulement (lent : pense à activer le GPU)"
print("Matériel détecté :", materiel)

## Les données et la règle du jeu

On charge le corpus Allociné (même source qu'au J3), on convertit l'étiquette en entier (`0` = négatif, `1` = positif) et on sous-échantillonne pour que tout tienne en quelques minutes. Un seul **jeu de test** sert à comparer tous les modèles.

In [ ]:
URL = "https://raw.githubusercontent.com/mickaeltemporao/lillelms/main/data/critiques_films"
train = pd.read_csv(f"{URL}/critiques_films_train.csv")
test = pd.read_csv(f"{URL}/critiques_films_test.csv")

mapping = {"négatif": 0, "positif": 1}
train["label"] = train["polarite"].map(mapping)
test["label"] = test["polarite"].map(mapping)

# Sous-échantillonnage (ajuste à la hausse si tu as le temps / un bon GPU)
train = train.sample(2000, random_state=42).reset_index(drop=True)
test = test.sample(800, random_state=42).reset_index(drop=True)
y_train, y_test = train["label"].to_numpy(), test["label"].to_numpy()
print(f"Train : {len(train)} | Test : {len(test)}")

# On collecte ici les scores de chaque modèle pour le tableau final.
resultats = {}
def evaluer(nom, y_vrai, y_pred, n=None):
    acc, f1 = accuracy_score(y_vrai, y_pred), f1_score(y_vrai, y_pred)
    resultats[nom] = {"accuracy": round(acc, 3), "f1": round(f1, 3), "n_test": n or len(y_vrai)}
    print(f"{nom:46} acc={acc:.3f}  f1={f1:.3f}")

## Modèle 1 : comptes de mots + régression logistique

Le point de départ historique du text mining : on compte les mots (sac de mots, sans nettoyage ni pondération) et on entraîne une régression logistique.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

cv = CountVectorizer()                       # tokenisation basique, comptes bruts
Xtr = cv.fit_transform(train["text"]); Xte = cv.transform(test["text"])

m1 = LogisticRegression(max_iter=1000).fit(Xtr, y_train)
evaluer("1. Comptes + régression log", y_test, m1.predict(Xte))

## Modèles 2 & 3 : nettoyer le vocabulaire (comme au J3) puis TF-IDF

Plutôt que de garder tous les mots, on **réduit le vocabulaire** exactement dans l'esprit du notebook `01-text-mining-concepts` :
- **lemmatisation** (ramener « adoré », « adore », « adorais » à `adorer`) ;
- **retrait des stopwords** français (mots outils : « le », « de », « et »...) ;
- **filtrage de fréquence** (on ignore les mots trop rares ou trop courants) via `min_df` / `max_df`.

On utilise **spaCy** (`fr_core_news_sm`) pour la lemmatisation et les stopwords français, puis `TfidfVectorizer` pour la pondération et le filtrage.

In [ ]:
import spacy
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])

def nettoyer(textes):
    # lemmatise, retire stopwords / ponctuation / mots <= 2 lettres
    sorties = []
    for doc in nlp.pipe(textes, batch_size=64):
        lemmes = [t.lemma_.lower() for t in doc
                  if t.is_alpha and not t.is_stop and len(t.lemma_) > 2]
        sorties.append(" ".join(lemmes))
    return sorties

train_clean = nettoyer(train["text"].tolist())
test_clean = nettoyer(test["text"].tolist())
print("Exemple nettoyé :", train_clean[0][:120], "...")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# min_df=5 : mot présent dans >= 5 critiques ; max_df=0.5 : on jette les mots trop fréquents
tfidf = TfidfVectorizer(min_df=5, max_df=0.5)
Ttr = tfidf.fit_transform(train_clean); Tte = tfidf.transform(test_clean)
print("Vocabulaire après réduction :", len(tfidf.get_feature_names_out()), "mots")

**Modèle 2** : TF-IDF + régression logistique.

In [ ]:
m2 = LogisticRegression(max_iter=1000).fit(Ttr, y_train)
evaluer("2. TF-IDF nettoyé + régression log", y_test, m2.predict(Tte))

**Modèle 3** : mêmes features, mais un **random forest** (des centaines d'arbres de décision qui votent).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

m3 = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Ttr, y_train)
evaluer("3. TF-IDF nettoyé + random forest", y_test, m3.predict(Tte))

## Modèles 4 à 6 : DistilCamemBERT comme extracteur de features

Idée commune : un transformer transforme chaque critique en un **vecteur** (son *embedding*, ici la moyenne des états cachés). On entraîne ensuite un **random forest** sur ces vecteurs.

Ce qui change d'un modèle à l'autre, c'est **combien on adapte l'encodeur** à notre tâche avant d'extraire les vecteurs :
- **4. gelé** : on ne touche pas au transformer (transfert pur) ;
- **5. fine-tuning partiel** : on ré-entraîne seulement ses **couches hautes** ;
- **6. fine-tuning complet** : on ré-entraîne **tout** l'encodeur.

Plus on adapte, meilleurs (en principe) sont les vecteurs pour *cette* tâche, mais plus c'est coûteux.

In [ ]:
from transformers import AutoTokenizer, AutoModel
MODELE = "cmarkea/distilcamembert-base"   # DistilCamemBERT : un CamemBERT léger et rapide
tokenizer = AutoTokenizer.from_pretrained(MODELE)
appareil = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def embeddings(encodeur, textes, batch=32, max_len=256):
    """Vecteur par texte = moyenne (pondérée par le masque) des états cachés."""
    encodeur = encodeur.to(appareil).eval()
    vecteurs = []
    for i in range(0, len(textes), batch):
        lot = tokenizer(textes[i:i+batch], padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt").to(appareil)
        sortie = encodeur(**lot).last_hidden_state            # (n, tokens, dim)
        masque = lot["attention_mask"].unsqueeze(-1).float()
        moyenne = (sortie * masque).sum(1) / masque.sum(1).clamp(min=1)
        vecteurs.append(moyenne.cpu().numpy())
    return np.vstack(vecteurs)

### Modèle 4 : encodeur gelé + random forest

In [ ]:
encodeur_gele = AutoModel.from_pretrained(MODELE)
Etr = embeddings(encodeur_gele, train["text"].tolist())
Ete = embeddings(encodeur_gele, test["text"].tolist())

m4 = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Etr, y_train)
evaluer("4. DistilCamemBERT gelé + RF", y_test, m4.predict(Ete))

### Fine-tuning (modèles 5 et 6)

On définit un petit utilitaire qui fine-tune un DistilCamemBERT de classification, puis renvoie son **encodeur adapté** (pour en réextraire des embeddings). Le paramètre `n_couches` contrôle l'ampleur :
- `n_couches=1` : seules la **dernière couche** et la tête sont ré-entraînées (partiel) ;
- `n_couches=None` : **tout** est ré-entraîné (complet).

In [ ]:
import re
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from datasets import Dataset

def encoder_lot(lot):
    return tokenizer(lot["text"], truncation=True, max_length=256)

ds_train = Dataset.from_pandas(train[["text", "label"]]).map(encoder_lot, batched=True)
ds_test = Dataset.from_pandas(test[["text", "label"]]).map(encoder_lot, batched=True)

def geler_sauf_top(model, n_couches):
    """Ne laisse entraînables que les `n_couches` couches hautes + la tête."""
    if n_couches is None:
        return  # fine-tuning complet : tout reste entraînable
    noms = [n for n, _ in model.named_parameters()]
    idx = sorted({int(m.group(1)) for n in noms
                  for m in [re.search(r"\.layer\.(\d+)\.", n)] if m})
    a_entrainer = set(idx[-n_couches:])
    for n, p in model.named_parameters():
        m = re.search(r"\.layer\.(\d+)\.", n)
        couche_haute = bool(m) and int(m.group(1)) in a_entrainer
        tete = any(k in n for k in ["classifier", "pooler"])
        p.requires_grad = couche_haute or tete

def fine_tuner(n_couches):
    model = AutoModelForSequenceClassification.from_pretrained(MODELE, num_labels=2)
    geler_sauf_top(model, n_couches)
    args = TrainingArguments(output_dir="ft", num_train_epochs=1,
                             per_device_train_batch_size=16, learning_rate=2e-5,
                             weight_decay=0.01, logging_steps=50, report_to="none")
    Trainer(model=model, args=args, train_dataset=ds_train,
            data_collator=DataCollatorWithPadding(tokenizer)).train()
    return model.base_model  # l'encodeur adapté

### Modèle 5 : fine-tuning partiel + random forest

In [ ]:
enc_partiel = fine_tuner(n_couches=1)
Etr5 = embeddings(enc_partiel, train["text"].tolist())
Ete5 = embeddings(enc_partiel, test["text"].tolist())

m5 = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Etr5, y_train)
evaluer("5. DistilCamemBERT partiel + RF", y_test, m5.predict(Ete5))

### Modèle 6 : fine-tuning complet + random forest

In [ ]:
enc_complet = fine_tuner(n_couches=None)
Etr6 = embeddings(enc_complet, train["text"].tolist())
Ete6 = embeddings(enc_complet, test["text"].tolist())

m6 = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Etr6, y_train)
evaluer("6. DistilCamemBERT complet + RF", y_test, m6.predict(Ete6))

## Modèle 7 : un LLM qui raisonne (*chain-of-thought*)

Dernière approche, radicalement différente : on **ne l'entraîne pas du tout**. On demande à un grand modèle de langue (`gpt-4o-mini`) de **raisonner étape par étape** sur la critique (le ton, les mots positifs / négatifs, l'avis global) avant de conclure. C'est le *chain-of-thought* : forcer le modèle à « réfléchir à voix haute » améliore souvent ses décisions.

Les appels API étant lents et facturés, on évalue le LLM sur un **sous-échantillon** du test (les autres modèles, eux, voient tout le test).

In [ ]:
from openai import OpenAI

# La clé est communiquée en classe. Colle-la entre les guillemets.
api_key = "COLLE_TA_CLE_ICI"
client = OpenAI(api_key=api_key)

SYSTEME = "Tu es un annotateur de sentiment pour des critiques de films en français."

def prompt_cot(critique):
    return (f'Critique :\n"""{critique[:1500]}"""\n\n'
            "Raisonne brièvement, étape par étape : le ton général, les mots positifs "
            "ou négatifs, l'avis d'ensemble. Puis conclus.\n"
            "Termine IMPÉRATIVEMENT par une dernière ligne au format exact :\n"
            "RÉPONSE: positif    (ou)    RÉPONSE: négatif")

def parser_label(texte):
    t = texte.lower()
    pos, neg = t.rfind("positif"), t.rfind("négatif")
    if pos == neg:      # aucun des deux trouvé
        return 0
    return 1 if pos > neg else 0

def scorer_llm(critique):
    rep = client.chat.completions.create(
        model="gpt-4o-mini", temperature=0, max_tokens=250,
        messages=[{"role": "system", "content": SYSTEME},
                  {"role": "user", "content": prompt_cot(critique)}])
    return parser_label(rep.choices[0].message.content)

In [ ]:
# Sous-échantillon pour limiter le coût (≈ 150 appels)
sous = test.sample(150, random_state=0).reset_index(drop=True)
y_pred_llm = [scorer_llm(t) for t in sous["text"]]

evaluer("7. LLM chain-of-thought (gpt-4o-mini)", sous["label"].to_numpy(),
        np.array(y_pred_llm), n=len(sous))

## Le verdict : tableau comparatif

On rassemble accuracy et F1 de tous les modèles.

In [ ]:
compar = pd.DataFrame(resultats).T.sort_values("f1", ascending=False)
compar

In [ ]:
ax = compar[["accuracy", "f1"]].plot(kind="barh", figsize=(9, 5))
ax.set_xlabel("score"); ax.set_xlim(0, 1)
ax.invert_yaxis(); ax.set_title("Comparaison des 7 approches (test Allociné)")
plt.tight_layout(); plt.show()

## Ce qu'on observe (en général)

- Les **baselines classiques** (1-3) sont étonnamment solides sur une tâche de sentiment binaire : sac de mots et TF-IDF capturent beaucoup de signal, pour un coût dérisoire.
- L'**encodeur gelé** (4) n'est pas toujours meilleur que TF-IDF : un transformer non adapté ne bat pas forcément un bon vieux comptage de mots.
- Le **fine-tuning** (5-6) est là que les transformers prennent l'avantage : plus on adapte l'encodeur, meilleurs sont les embeddings pour *notre* tâche.
- Le **LLM** (7) atteint souvent un excellent score **sans aucun entraînement**, mais il est **lent, facturé**, et c'est une grosse boîte noire pour une tâche qu'un modèle minuscule règle très bien.

**La leçon** : le modèle le plus sophistiqué n'est pas toujours le bon choix. On arbitre entre **performance**, **coût**, **rapidité** et **interprétabilité** selon le problème. Pour classer du sentiment binaire, un TF-IDF suffit souvent ; on réserve le fine-tuning et les LLM aux tâches où le contexte fin est vraiment décisif.